In [1]:
%pip install pandas bert-score nltk

  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinu

In [1]:
import pandas as pd
from bert_score import score
from nltk import ngrams

/home/pranesh/Desktop/Counter narrative generation task/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
import pandas as pd
from bert_score import score
from nltk import ngrams


# ---------- READ PREDICTIONS SAFELY ----------
def read_prediction_file(file_path):

    predictions = []

    with open(file_path, "r", encoding="utf-8") as f:

        lines = f.readlines()

        for line in lines[1:]:   # skip header

            line = line.strip()

            if not line:
                continue

            # split only first comma
            parts = line.split(",", 1)

            if len(parts) == 2:
                predictions.append(parts[1].strip())

    return predictions


# ---------- READ GOLD FILE SAFELY ----------
def read_gold_file(file_path):

    references = []

    with open(file_path, "r", encoding="utf-8") as f:

        lines = f.readlines()

        for line in lines[1:]:  # skip header

            parts = line.strip().split(",", 4)

            if len(parts) >= 5:
                references.append(parts[4].strip())

    return references


# ---------- DISTINCT-2 ----------
def distinct2(sentences):

    all_bigrams = []

    for s in sentences:

        tokens = str(s).split()

        if len(tokens) < 2:
            continue

        bigrams = list(ngrams(tokens, 2))
        all_bigrams.extend(bigrams)

    unique_bigrams = set(all_bigrams)

    if len(all_bigrams) == 0:
        return 0

    return len(unique_bigrams) / len(all_bigrams)


# ---------- BERTSCORE ----------
def compute_bertscore(preds, refs):

    P, R, F1 = score(preds, refs, lang="en")

    return F1.mean().item()


# ---------- EVALUATION ----------
def evaluate(pred_file, gold_file):

    predictions = read_prediction_file(pred_file)
    references = read_gold_file(gold_file)

    min_len = min(len(predictions), len(references))

    predictions = predictions[:min_len]
    references = references[:min_len]

    bert = compute_bertscore(predictions, references)
    d2 = distinct2(predictions)

    print("\n==============================")
    print("File:", pred_file.split("/")[-1])
    print("Samples evaluated:", min_len)
    print("BERTScore:", round(bert,4), "(", round(bert*100,2),"%)")
    print("Distinct-2:", round(d2,4), "(", round(d2*100,2),"%)")
    print("==============================\n")



# ---------- RUN EVALUATION ----------
evaluate(
"/home/pranesh/Desktop/Counter narrative generation task/JusticeBots_Task2/JusticeBots_Tamil_gpt.csv",
"/home/pranesh/Desktop/Counter narrative generation task/CN_Tamil_Test.csv"
)

evaluate(
"/home/pranesh/Desktop/Counter narrative generation task/JusticeBots_Task2/JusticeBots_Tamil_perplexity.csv",
"/home/pranesh/Desktop/Counter narrative generation task/CN_Tamil_Test.csv"
)

evaluate(
"/home/pranesh/Desktop/Counter narrative generation task/JusticeBots_Task2/JusticeBots_English_gpt.csv",
"/home/pranesh/Desktop/Counter narrative generation task/CN_English_Test.csv"
)

evaluate(
"/home/pranesh/Desktop/Counter narrative generation task/JusticeBots_Task2/JusticeBots_English_perplexity.csv",
"/home/pranesh/Desktop/Counter narrative generation task/CN_English_Test.csv"
)

Loading weights: 100%|██████████| 389/389 [00:01<00:00, 214.08it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



File: JusticeBots_Tamil_gpt.csv
Samples evaluated: 109
BERTScore: 0.9421 ( 94.21 %)
Distinct-2: 0.7517 ( 75.17 %)



Loading weights: 100%|██████████| 389/389 [00:02<00:00, 194.07it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



File: JusticeBots_Tamil_perplexity.csv
Samples evaluated: 109
BERTScore: 0.9493 ( 94.93 %)
Distinct-2: 0.6313 ( 63.13 %)



Loading weights: 100%|██████████| 389/389 [00:01<00:00, 199.91it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



File: JusticeBots_English_gpt.csv
Samples evaluated: 66
BERTScore: 0.8792 ( 87.92 %)
Distinct-2: 0.7429 ( 74.29 %)



Loading weights: 100%|██████████| 389/389 [00:01<00:00, 208.45it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



File: JusticeBots_English_perplexity.csv
Samples evaluated: 66
BERTScore: 0.8693 ( 86.93 %)
Distinct-2: 0.8957 ( 89.57 %)



In [13]:
import pandas as pd
from bert_score import score
from nltk import ngrams


# ---------- READ PREDICTIONS SAFELY ----------
def read_prediction_file(file_path):

    predictions = []

    with open(file_path, "r", encoding="utf-8") as f:

        lines = f.readlines()

        for line in lines[1:]:   # skip header

            line = line.strip()

            if not line:
                continue

            # split only first comma
            parts = line.split(",", 1)

            if len(parts) == 2:
                predictions.append(parts[1].strip())

    return predictions


# ---------- READ GOLD FILE SAFELY ----------
def read_gold_file(file_path):

    references = []

    with open(file_path, "r", encoding="utf-8") as f:

        lines = f.readlines()

        for line in lines[1:]:  # skip header

            parts = line.strip().split(",", 4)

            if len(parts) >= 5:
                references.append(parts[4].strip())

    return references


# ---------- DISTINCT-2 ----------
def distinct2(sentences):

    all_bigrams = []

    for s in sentences:

        tokens = str(s).split()

        if len(tokens) < 2:
            continue

        bigrams = list(ngrams(tokens, 2))
        all_bigrams.extend(bigrams)

    unique_bigrams = set(all_bigrams)

    if len(all_bigrams) == 0:
        return 0

    return len(unique_bigrams) / len(all_bigrams)


# ---------- BERTSCORE ----------
def compute_bertscore(preds, refs):

    P, R, F1 = score(preds, refs, lang="en")

    return F1.mean().item()


# ---------- EVALUATION ----------
def evaluate(pred_file, gold_file):

    predictions = read_prediction_file(pred_file)
    references = read_gold_file(gold_file)

    min_len = min(len(predictions), len(references))

    predictions = predictions[:min_len]
    references = references[:min_len]

    bert = compute_bertscore(predictions, references)
    d2 = distinct2(predictions)

    print("\n==============================")
    print("File:", pred_file.split("/")[-1])
    print("Samples evaluated:", min_len)
    print("BERTScore:", round(bert,4), "(", round(bert*100,2),"%)")
    print("Distinct-2:", round(d2,4), "(", round(d2*100,2),"%)")
    print("==============================\n")



# ---------- RUN EVALUATION ----------
evaluate(
"/home/pranesh/Desktop/Counter narrative generation task/JusticeBots_Task2/JusticeBots_English_gemini.csv",
"/home/pranesh/Desktop/Counter narrative generation task/CN_English_Test.csv"
)

evaluate(
"/home/pranesh/Desktop/Counter narrative generation task/JusticeBots_Task2/JusticeBots_Tamil_gemini.csv",
"/home/pranesh/Desktop/Counter narrative generation task/CN_Tamil_Test.csv"
)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 408.16it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



File: JusticeBots_English_gemini.csv
Samples evaluated: 66
BERTScore: 0.8727 ( 87.27 %)
Distinct-2: 0.6815 ( 68.15 %)



Loading weights: 100%|██████████| 389/389 [00:00<00:00, 424.77it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



File: JusticeBots_Tamil_gemini.csv
Samples evaluated: 109
BERTScore: 0.9515 ( 95.15 %)
Distinct-2: 0.9151 ( 91.51 %)

